# Vast.ai training

Upload this notebook to `/workspace` on your single-GPU **NVIDIA CUDA** instance. Set your Hugging Face write token below (or supply `HF_TOKEN` in the instance environment), then **Run All**.

It clones the source, installs dependencies, downloads and extracts `vizwiz.zip` into `vizwiz/`, deletes the ZIP to free space, and runs the **13 experiments sequentially, for 30 epochs each**. Checkpoints and final metrics upload to your public Hugging Face repository **after each completed run**. `RUN_IDS = []` runs the whole study.

Run All again to reuse the dataset and continue from saved epochs; completed runs are skipped and failed uploads are retried. Completed Hub backups can also be restored on a new instance. Execution stays in this notebook; interrupting the run cell stops its processes. Local work is under `/workspace/cm-study`.


In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Or paste your write token here; keep the notebook private.
RUN_IDS = []  # All 13 runs. Example: ["cm_shared_s42", "cm_shared_s123", "cm_shared_s456"]

WORK_DIR = "/workspace/cm-study"
GIT_URL = "https://github.com/DPTPhatUS/VizWiz-VQA-Grounding.git"
GIT_BRANCH = "exp/context-membership"
HF_REPO = "Last-Bullet/vizwiz-cm-aciids-2027"
HF_PREFIX = "matched-30ep-v1"  # Choose a new prefix for a separate study.


In [ ]:
from pathlib import Path
import signal
import subprocess
import sys

work = Path(WORK_DIR)
work.mkdir(parents=True, exist_ok=True)
repo = work / "source"
env = os.environ.copy()
env.update(HF_HOME=str(work / "cache/huggingface"), UV_CACHE_DIR=str(work / "cache/uv"),
           MPLBACKEND="Agg", OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false",
           PYTHONUNBUFFERED="1")
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

def run(command, cwd=None):
    process = subprocess.Popen([str(part) for part in command], cwd=cwd, env=env, start_new_session=True)
    try:
        code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
        raise
    if code:
        raise subprocess.CalledProcessError(code, command)

if not repo.exists():
    run(["git", "clone", "--single-branch", "--branch", GIT_BRANCH, GIT_URL, repo])
run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
run([sys.executable, "-m", "uv", "python", "install", "3.12"])
run([sys.executable, "-m", "uv", "sync", "--frozen", "--python", "3.12"], cwd=repo)

command = [repo / ".venv/bin/python", "-u", "-m", "experiments.vast", "--work-dir", work,
           "--hf-repo", HF_REPO, "--hf-prefix", HF_PREFIX]
if RUN_IDS:
    command += ["--runs", *RUN_IDS]
run(command, cwd=repo)
